# Prepare the data (Zacharie Houle)

The project rubric draws a sharp line between *Developing (11 to 20 pts)* ("Preprocessing done without rationale and in-depth explanation") and *Excellent (21 to 25 pts)* ("Cleaning, feature engineering, and scaling justified and automated in a leak-free pipeline, separated train/test").

To meet the *Excellent* standard, every step in this notebook follows a three-part pattern:
1. Empirical check on `train_df` (never on validation or test)
2. Written rationale explaining why we chose a specific transformation over the alternatives taught in Modules 2, 3, and 4
3. Implementation inside a modular function or scikit-learn transformer in `src/preprocessing.py` so the choice can also be tuned as a hyperparameter in Notebook 04

In [ ]:
import shutil
import sys
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import set_config
from sklearn.model_selection import train_test_split

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import (
    CollinearityDropper,
    QuantileOutlierClipper,
    build_preprocessing_pipeline,
    decompose_datetime_features,
    drop_uninformative_columns,
)

set_config(display="diagram")
pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42

RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
EXTERNAL_DATA_DIR = REPO_ROOT / "data" / "external"
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"

## 0. Checklist notes: working on copies, splitting first, and writing modular functions

### Why we split before any preprocessing
If we compute medians, quantile bounds, correlation matrices, or mean/standard deviation on the full dataset before splitting, information from the validation and test sets leaks into training. That produces overly optimistic validation scores and train/serve skew in production. We therefore split `data/raw/train.csv` immediately into `train_df` (70%), `val_df` (15%), and `test_df` (15%).

### Why every transformation is written as a function or transformer
Following Géron (Appendix B, p. 757), writing functions and `BaseEstimator`/`TransformerMixin` classes in `src/preprocessing.py` serves five purposes:
1. We can prepare a fresh dataset in one call the next time data arrives.
2. We can reuse the same transformer classes in future projects.
3. We can transform `val_df` and `test_df` using only parameters learned on `train_df`.
4. We can score new live instances in production with identical logic.
5. We can treat uncertain preparation choices (such as whether to clip outliers or which imputer to use) as hyperparameters during cross-validation in Notebook 04.

In [ ]:
# 1. Download raw dataset via kagglehub if not already present in data/raw/
raw_csv_path = RAW_DATA_DIR / "train.csv"
if not raw_csv_path.exists():
    dl_dir = Path(kagglehub.competition_download("actuarial-loss-estimation"))
    shutil.copy(dl_dir / "train.csv", raw_csv_path)

# Work on a copy of the original dataset (keep raw file intact)
raw_df = pd.read_csv(raw_csv_path)
data_copy = raw_df.copy()

# 2. Split into train (70%), val (15%), and held-out test (15%) BEFORE any fitting
train_val_df, test_df = train_test_split(
    data_copy, test_size=0.15, random_state=RANDOM_STATE
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.15 / 0.85, random_state=RANDOM_STATE
)

print(f"Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")

## 1. Data cleaning

### 1.1 Fix or remove outliers: rationale and trade-offs

When handling extreme values, we have three options: drop the rows, leave the values untouched, or cap (winsorize) them at empirical quantiles.

Dropping extreme rows is rejected for two reasons. First, in production scoring we cannot refuse to predict on a new record just because one of its inputs is extreme. Second, in financial and actuarial data, extreme predictors often correspond to the highest-cost cases we care most about predicting.

Leaving all extremes untouched works for tree-based models (which split on rank thresholds), but it distorts ordinary least squares, Ridge/Lasso coefficients, and `StandardScaler` variance for linear and neural models. Furthermore, some extremes are physical recording errors (for example, `HoursWorkedPerWeek` exceeding the 168 hours in a 7-day week).

We therefore apply two levels of outlier treatment:
1. Domain sanity rules for impossible values (such as capping weekly hours at 168 and setting negative reporting delays to 0).
2. Quantile clipping via `QuantileOutlierClipper` (1st and 99th percentiles learned strictly on `train_df`), with `clip_outliers=True/False` exposed as a tunable hyperparameter so Notebook 04 can verify whether clipping improves cross-validated error.

In [ ]:
# Empirical check on train_df: compare min, 1st percentile, median, 99th percentile, and max
num_preview_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
outlier_audit = train_df[num_preview_cols].quantile([0.0, 0.01, 0.50, 0.99, 1.0]).T
outlier_audit.columns = ["min", "p01", "median", "p99", "max"]
outlier_audit

### 1.2 Fill in missing values: rationale and trade-offs

We must choose between four strategies for missing values: dropping rows/columns, filling with zero/constant, univariate imputation (mean vs. median), or multivariate imputation (`IterativeImputer`), alongside whether to add a missingness flag.

1. Why we add a boolean missingness indicator (`add_indicator=True`): As covered in Module 3 (§2.4), missingness in administrative and financial records is rarely completely random (MCAR). An unreported wage, marital status, or work schedule often reflects part-time, casual, or disputed employment. Adding a binary `_na` flag preserves that signal before filling the gap.
2. Why we reject dropping rows: Dropping rows with missing predictors shrinks the training sample and prevents the model from scoring incomplete records at inference time.
3. Why we prefer `median` and `IterativeImputer` over `mean`: Because financial and employment variables are heavily right-skewed, the sample mean is pulled upward by outliers, whereas the median represents a typical observation. When features are correlated (for example, `WeeklyWages` and `HoursWorkedPerWeek`), `IterativeImputer` (Module 2) predicts the missing value from the other observed features rather than plugging in a flat constant. Both `median` and `iterative` are exposed in `build_preprocessing_pipeline(num_imputer_strategy=...)` so Notebook 04 can compare them via cross-validation.

In [ ]:
# Audit missing value counts and percentages on train_df
missing_summary = pd.DataFrame({
    "missing_count": train_df.isna().sum(),
    "missing_pct": (train_df.isna().mean() * 100).round(2),
    "dtype": train_df.dtypes.astype(str),
})
missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_pct", ascending=False)

## 2. Feature selection

### 2.1 Drop attributes that provide no useful information: rationale and trade-offs

Following Module 4 (*Dimensionality Reduction & Feature Selection*), keeping irrelevant or collinear columns hurts both interpretability and generalization: irrelevant columns add variance, while collinear columns ($|r| \ge 0.95$) destabilize linear regression weights and split tree feature importance across duplicate variables.

We choose feature selection over dimensionality reduction (PCA/UMAP) for our tabular predictors because our business objective requires feature-level explainability (knowing whether reporting delay or initial reserve drives the prediction). We apply two selection filters:
1. Deterministic removal of administrative identifiers (`ClaimNumber`) that have 100% cardinality and zero generalizable signal.
2. Automated collinearity pruning via `CollinearityDropper(threshold=0.95)` fitted strictly on `train_df`.

In [ ]:
UNINFORMATIVE_COLS = ["ClaimNumber"]

train_clean = drop_uninformative_columns(train_df, UNINFORMATIVE_COLS)
val_clean = drop_uninformative_columns(val_df, UNINFORMATIVE_COLS)
test_clean = drop_uninformative_columns(test_df, UNINFORMATIVE_COLS)
print(f"Columns after dropping identifiers: {train_clean.shape[1]}")

## 3. Feature engineering

Raw administrative fields rarely match the functional form a model needs. Below we justify each of the four feature engineering operations required by Géron's checklist and Module 3.

### 3.1 Decompose features (dates and categoricals)
A raw ISO timestamp (`DateTimeOfAccident`, `DateReported`) cannot be fed into a regression model directly. Decomposing each date into `year`, `month`, `day`, `dayofweek`, and `is_weekend` captures two distinct domain effects: inflation/policy drift across years, and Monday-morning reporting spikes or weekend shift severity.

For categorical features (`Gender`, `MaritalStatus`, `PartTimeFullTime`), we consult the Module 3 encoding decision table (§2.3.12). Because these variables are nominal with low cardinality (2 to 4 unique values) and have no natural order, One-Hot Encoding is chosen over Label or Ordinal Encoding to avoid injecting a false numerical ranking into linear and neural models.

### 3.2 Aggregate features into promising new features
Domain knowledge tells us that two raw variables often interact to form the true economic driver:
1. `reporting_delay_days` (`DateReported - DateTimeOfAccident`): Claims reported weeks after the accident often involve cumulative trauma, soft-tissue ambiguity, or legal disputes, making them much harder for human adjusters to price on day one.
2. `hourly_wage` (`WeeklyWages / HoursWorkedPerWeek`): Two workers earning $600/week have very different profiles if one works 15 hours and the other works 60 hours.
3. `hours_per_day` (`HoursWorkedPerWeek / DaysWorkedPerWeek`): Measures shift length and fatigue risk.
4. `total_dependents` (`DependentChildren + DependentsOther`): Captures total household financial pressure, which influences return-to-work incentives.

### 3.3 Add promising mathematical transformations ($\log(1+x)$, $\sqrt{x}$, `Yeo-Johnson`)
Financial variables (`InitialIncurredCalimsCost`, `WeeklyWages`, `hourly_wage`) and `reporting_delay_days` exhibit strong positive skewness (> 2.0). In linear models, neural networks, and distance-based algorithms, a few extreme dollar observations dominate the loss gradient. Applying $\log(1+x)$ or `Yeo-Johnson` compresses the right tail toward a Gaussian shape and turns multiplicative relationships into additive linear ones.

### 3.4 Discretize continuous features (`KBinsDiscretizer`)
Worker `Age` has a non-linear effect on recovery time (younger workers have higher accident frequency but faster physical recovery, whereas workers over 55 face longer rehabilitation plateaus). Adding quantile-discretized age bins alongside continuous age lets linear models capture step-function risk brackets without requiring manual spline knots.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Apply stateless feature decomposition, domain sanity caps, and aggregations on a copy of df."""
    out = df.copy()

    # Domain sanity cap on weekly hours (max 168 hours in a 7-day week)
    if "HoursWorkedPerWeek" in out.columns:
        out["HoursWorkedPerWeek"] = out["HoursWorkedPerWeek"].clip(lower=0, upper=168)

    # Decompose and aggregate timestamps
    if "DateTimeOfAccident" in out.columns and "DateReported" in out.columns:
        acc_dt = pd.to_datetime(out["DateTimeOfAccident"], errors="coerce")
        rep_dt = pd.to_datetime(out["DateReported"], errors="coerce")
        out["reporting_delay_days"] = (rep_dt - acc_dt).dt.days.clip(lower=0)
        out = decompose_datetime_features(
            out, date_cols=["DateTimeOfAccident", "DateReported"], drop_original=True
        )

    # Aggregate employment and demographic features into domain ratios
    if "WeeklyWages" in out.columns and "HoursWorkedPerWeek" in out.columns:
        hours = out["HoursWorkedPerWeek"].replace(0, np.nan)
        out["hourly_wage"] = out["WeeklyWages"] / hours

    if "HoursWorkedPerWeek" in out.columns and "DaysWorkedPerWeek" in out.columns:
        days = out["DaysWorkedPerWeek"].replace(0, np.nan)
        out["hours_per_day"] = out["HoursWorkedPerWeek"] / days

    if "DependentChildren" in out.columns and "DependentsOther" in out.columns:
        out["total_dependents"] = (
            out["DependentChildren"].fillna(0) + out["DependentsOther"].fillna(0)
        )

    return out

train_eng = engineer_features(train_clean)
val_eng = engineer_features(val_clean)
test_eng = engineer_features(test_clean)
print(f"Columns after feature engineering: {train_eng.shape[1]}")

In [ ]:
# Empirical verification of skewness reduction on train_eng before vs. after log1p
SKEWED_CANDIDATES = [
    c for c in ["InitialIncurredCalimsCost", "WeeklyWages", "hourly_wage", "reporting_delay_days"]
    if c in train_eng.columns
]
skew_check = pd.DataFrame({
    "raw_skewness": train_eng[SKEWED_CANDIDATES].skew().round(2),
    "log1p_skewness": np.log1p(train_eng[SKEWED_CANDIDATES].clip(lower=0)).skew().round(2),
})
skew_check

## 4. Feature scaling: rationale and trade-offs

Following Module 3 (§2.1 and §2.2), we must choose between Standardization (`StandardScaler`), Normalization (`MinMaxScaler`), and Robust Scaling (`RobustScaler`):

| Method | Formula | When to use | Why chosen or rejected here |
|---|---|---|---|
| `MinMaxScaler` | $(x - \min) / (\max - \min)$ | Bounded inputs with fixed range (such as image pixels 0 to 255) | Rejected as default because a single extreme wage or reserve crushes all normal observations into a tiny interval near 0. |
| `StandardScaler` | $(x - \mu) / \sigma$ | Unbounded continuous variables (especially after log transformation) | Chosen as default so regularized linear models (Ridge/Lasso), SVMs, and neural nets receive zero-mean, unit-variance inputs. |
| `RobustScaler` | $(x - \text{median}) / \text{IQR}$ | Heavy-tailed variables when outlier clipping is disabled | Exposed as a tunable hyperparameter (`scaler_type="robust"`) for Notebook 04. |

Note that tree-based models (Random Forest, LightGBM, XGBoost) split on rank order and are invariant to monotonic scaling, but standardizing the shared dataset allows every model family in Notebook 03 (linear, neural, and tree-based) to be trained and compared on the exact same feature matrix.

In [ ]:
TARGET_COL = "UltimateIncurredClaimCost"
TEXT_COL = "ClaimDescription"

SKEWED_NUMERIC_COLS = SKEWED_CANDIDATES
DISCRETIZE_COLS = [c for c in ["Age"] if c in train_eng.columns]
LOW_CARD_CAT_COLS = [
    c for c in ["Gender", "MaritalStatus", "PartTimeFullTime"]
    if c in train_eng.columns
]
NUMERIC_COLS = [
    c for c in train_eng.select_dtypes(include=[np.number]).columns
    if c not in SKEWED_NUMERIC_COLS + [TARGET_COL]
]

prep_pipeline = build_preprocessing_pipeline(
    numeric_cols=NUMERIC_COLS,
    skewed_numeric_cols=SKEWED_NUMERIC_COLS,
    discretize_cols=DISCRETIZE_COLS,
    low_card_cat_cols=LOW_CARD_CAT_COLS,
    clip_outliers=True,
    num_imputer_strategy="median",
    add_missing_indicator=True,
    skew_transform="log1p",
    n_bins=5,
    scaler_type="standard",
    collinearity_threshold=0.95,
)

prep_pipeline

## 5. Fit on train, transform all splits, and export to `data/processed/`

We fit `prep_pipeline` strictly on `train_eng` (`fit_transform`), apply the learned parameters to `val_eng` and `test_eng` (`transform`), re-attach the target variable, and save `train.parquet`, `val.parquet`, and `test.parquet`.

In [ ]:
X_train_prep = prep_pipeline.fit_transform(train_eng.drop(columns=[TARGET_COL], errors="ignore"))
X_val_prep = prep_pipeline.transform(val_eng.drop(columns=[TARGET_COL], errors="ignore"))
X_test_prep = prep_pipeline.transform(test_eng.drop(columns=[TARGET_COL], errors="ignore"))

train_out = X_train_prep.assign(**{TARGET_COL: train_eng[TARGET_COL].values})
val_out = X_val_prep.assign(**{TARGET_COL: val_eng[TARGET_COL].values})
test_out = X_test_prep.assign(**{TARGET_COL: test_eng[TARGET_COL].values})

train_out.to_parquet(PROCESSED_DATA_DIR / "train.parquet", index=False)
val_out.to_parquet(PROCESSED_DATA_DIR / "val.parquet", index=False)
test_out.to_parquet(PROCESSED_DATA_DIR / "test.parquet", index=False)

print(f"Exported processed datasets to {PROCESSED_DATA_DIR}:")
print(f"  train.parquet: {train_out.shape}")
print(f"  val.parquet:   {val_out.shape}")
print(f"  test.parquet:  {test_out.shape}")